# 第 10 章 · 在想象里训练：BPTT 只看 32 步，critic 补上后面

配套讲义：仓库里 `robot-motion-rl/notes/lessons/10-bptt-critic.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（a、奖励、V 是编的，裁剪规则取自表 A.5）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 连乘 | $a^H$ | `jacobian_product(a, h)` | 1.1³² = 21.11；0.9³² = 0.0343 |
| 第二步 式 2 | 求和 + 折扣后的 V | `imagined_objective([1, 1, 1], 0.99, [1]*4, 50)` | 51.485 |
| 第三步 视野外 | — | `sees_reward(h, 40, critic)` | False、True、True |
| 第四步 裁剪 | 缩放 + 截断 | `clip_grad([3, 0.4, -0.05])` | [0.1, 0.1, −0.0165] |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `jacobian_product`**：一维时 dx_H/dx_0 = a^H：|a|>1 爆炸，|a|<1 消失。

In [ ]:
def jacobian_product(a, horizon):
    """一维时 dx_H/dx_0 = a^H：|a|>1 爆炸，|a|<1 消失。"""
    return a**horizon

**函数 `imagined_objective`**：式 2：Σ_{t<H} γ^t·c_t·r_{t+1} + γ^H·c_H·V(s_H)。survival 长度 H+1。

In [ ]:
def imagined_objective(rewards, gamma, survival, v_end):
    """式 2：Σ_{t<H} γ^t·c_t·r_{t+1} + γ^H·c_H·V(s_H)。survival 长度 H+1。"""
    h = len(rewards)
    j = sum(gamma**t * survival[t] * rewards[t] for t in range(h))
    return j + gamma**h * survival[h] * v_end

**函数 `sees_reward`**：视野 H 内看不到第 reward_step 步的奖励；有 critic 时 V(s_H) 能把它带回来（理想情况）。

In [ ]:
def sees_reward(horizon, reward_step, critic):
    """视野 H 内看不到第 reward_step 步的奖励；有 critic 时 V(s_H) 能把它带回来（理想情况）。"""
    return reward_step < horizon or critic

**函数 `clip_grad`**：SuperMimic 表 A.5：先按全局范数缩到 ≤ 1，再逐元素裁到 ±0.1。

In [ ]:
def clip_grad(g, global_max=1.0, elem_max=0.1):
    """SuperMimic 表 A.5：先按全局范数缩到 ≤ 1，再逐元素裁到 ±0.1。"""
    g = np.asarray(g, float)
    norm = np.linalg.norm(g)
    if norm > global_max:
        g = g * (global_max / norm)
    return np.clip(g, -elem_max, elem_max)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 雅可比连乘")
for a in (1.1, 0.9):
    for h in (8, 32, 128):
        print(f"a={a}, H={h}: a^H = {jacobian_product(a, h):.4g}")

In [ ]:
print("\n第二步 critic 补上视野外")
for h, critic in [(16, False), (16, True), (64, False)]:
    print(f"H={h}，critic={critic} → 看得到第 40 步的回报：{sees_reward(h, 40, critic)}")
j = imagined_objective([1, 1, 1], 0.99, [1, 1, 1, 1], 50.0)
print(f"H=3、每步奖励 1、V(s_H)=50、γ=0.99 → J = {j:.3f}")

In [ ]:
print("\n第三步 梯度裁剪")
print("g = [3, 0.4, −0.05] →", np.round(clip_grad([3, 0.4, -0.05]), 4))

## 3. 代码里才看得到的

- `sees_reward` 是个理想化的布尔判断：真实 critic 是学出来的，只能「大致」把视野外的价值带回来，学不准时策略仍然会短视。
- 式 2 里 $\hat r_{t+1}$ 的下标比 t 大 1（《从零学强化学习》第 1 章说过：做完动作才拿奖励），代码里用 `rewards[t]` 存的就是 $r_{t+1}$。

## 4. 自己改一改

- 把 V 改成 0（相当于没有 critic），看 J 掉多少，体会 critic 项的分量。
- 打印 1.1 和 0.9 在 H = 1…128 上的 $a^H$，画出爆炸和消失的曲线。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 多米诺连乘，H 长就爆炸或消失；
> 前三十二步精确求导，后面交给 critic 估；
> 没有 critic 只看眼前，推一把就站不稳；
> 确定策略没噪声可压，梯度先缩总量再截单个。